Tema 2 - Perceptron

In [1]:
import numpy as np
import math

train = np.load("./nn-2026-fii-homework-1/train.npz")
test = np.load("./nn-2026-fii-homework-1/test.npz")

X_train = train["data"]
y_train = train["labels"]
X_test = test["data"]

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)

print(y_train)

X_train = X_train / 255.0
X_test = X_test / 255.0


W = np.random.randn(784, 10) * 0.01
b = np.zeros(10)

V_w = np.zeros((784, 10)) #viteza pondere pt leraning rate momentum
V_b = np.zeros(10) # viteza pentru bias
beta = 0.9 # coeficient de frecare

(60000, 784)
(60000,)
(15000, 784)
[9 3 6 ... 9 5 7]


In [ ]:
from scipy.ndimage import rotate

X_2D = X_train.reshape(-1, 28, 28)

X_rotite_minus = []
X_rotite_plus = []

for img in X_2D[:10000]:
    X_rotite_minus.append(rotate(img, angle=-15, reshape=False))
    X_rotite_plus.append(rotate(img, angle=15, reshape=False))

# inapoi in vectori plati
X_minus_flat = np.array(X_rotite_minus).reshape(-1, 784)
X_plus_flat = np.array(X_rotite_plus).reshape(-1, 784)

X_train = np.concatenate((X_train, X_minus_flat, X_plus_flat), axis=0)

etichete_extra = y_train[:10000]
y_train = np.concatenate((y_train, etichete_extra, etichete_extra), axis=0)

Generam imagini augmentate... (va dura cateva secunde)


Calculam Z-ul (care va fi o matrice de (m,10), fiind scorul pentru fiecare 10 perceptroni la toate imaginile), si apoi il trecem prin softmax pentru a afla Y_prob (care va fi o matrice tot de (m,10), de data asta fiind probabilitatile pentru fiecare valoare 0-9, pentru fiecare imagine)

In [ ]:
def softmax(Z:np.array) -> np.array:
    Y = np.zeros((Z.shape[0],10))
    # line = 0
    # column = 0
    # for row in Z:
    #     ez_list = []
    #     for z in row:
    #         ez_list.append(math.exp(z))
    #     sum_ez = sum(ez_list)
    #     for z in row:
    #         yi = ez_list[column]/sum_ez
    #         Y[line][column] = yi
    #         column += 1
    #     line += 1
    #     column = 0
    
    exp_Z = np.exp(Z)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

# Y_hat va fi matrice de (m,1) si va stabili rezultatatele prezise la fiecare imagine
def predicted_class_matrix(Y:np.array) -> np.array:
    Y_hat = np.zeros((Y.shape[0],1))
    Y_hat = np.argmax(Y,axis=1,keepdims=True)
    return Y_hat
# Z will be of size (m,10)
def forward_propagation(X:np.array, W:np.array, b:np.array) -> np.array:
    Z = np.dot(X,W) + b
    # print(Z.shape)
    # print(Z)
    Y = softmax(Z)
    return Y

Y_prob = forward_propagation(X_train, W, b)
print(Y_prob)

Y_predicted = predicted_class_matrix(Y_prob)
print()
print(Y_predicted)


[[0.09181883 0.10570781 0.1116349  ... 0.09812499 0.09619208 0.09772239]
 [0.08981264 0.10995528 0.10388337 ... 0.10061409 0.09531271 0.094536  ]
 [0.09925785 0.10352923 0.10235272 ... 0.10440212 0.09811858 0.09825072]
 ...
 [0.08790821 0.10428872 0.11808777 ... 0.09882855 0.09210694 0.10146593]
 [0.0878824  0.09469801 0.10533149 ... 0.09875902 0.09911698 0.09733922]
 [0.1001908  0.09116084 0.10314894 ... 0.0887095  0.09565031 0.09509005]]

[[2]
 [6]
 [7]
 ...
 [2]
 [6]
 [6]]


Calculam CrossEntropy pentru a vedea eroarea. cross_entropy va returna eroarea calculata pentru toti cei 10 perceptroni (care este suma din produs vector target (care ar fi [0,0,1,0,...,0]) cu log(probabilitatea prezicerii de pe acel perceptron))

CrossEntropy este, pentru o imagine, suma din elementul din target(adica daca e cel prezis sau nu)*log(probabilitatea prezisa la acel index).

Dar pentru toate elementele mai putin target-ul meu corect, elementele din suma sunt zero, deci cross entropy ar fi -log(probabilitate_prezicere_corecta)

iar crossentropy pe tot setul vine -log(probabilitate_prezicere_corecta)/m

In [3]:
#putem sa ne folosim direct de y_train care e de forma [9,3,2,...., etc.] pentru a extrage indexul probabilitatii prezicerii corecte rapid
def cross_entropy(Y_prob:np.array, y_train:np.array)->np.array:
    m = y_train.shape[0]
    ## extrag valorile din y_prob de la 0 la m-1 la indicii din y_train
    probabilitati_corecte = Y_prob[np.arange(m), y_train]
    loss = -np.sum(np.log(probabilitati_corecte)) / m
    return loss

loss = cross_entropy(Y_prob, y_train)
print(loss)
    

2.315297637599279


Trecem la gradient descend. Vom calcula target, care este o matrice, fiecare linie avand forma [0,1,0,0,...,0], daca predictia mea ar trebui sa dea 1

Target-y va fi aceasta matrice minus matricea probabilitatilor, si da tot o matrice (m,10)

In [14]:
def get_target(y_train:np.array) -> np.array:
    Target = np.zeros((y_train.shape[0],10))
    # line=0
    # column=0
    # for label in y_train:
    #     for column in range(0,10):
    #         if column == label:
    #             Target[line][column] = 1
    #         else:
    #             Target[line][column] = 0
    #     line += 1

    Target[np.arange(y_train.shape[0]), y_train] = 1
    return Target

def gradient_descend(Target:np.array, Y_prob:np.array, learning_rate:float, X_train:np.array, W:np.array, b:np.array, V_w: np.ndarray, V_b: np.ndarray, beta: float):
    gradient = Target - Y_prob
    transpose = np.transpose(X_train)

    m = X_train.shape[0]

    change_w = (learning_rate * np.dot(transpose,gradient))/m

    #L2 Reg weight decay sa nu mai am overfit
    lambda_reg = 0.0001
    change_w = change_w - (learning_rate * lambda_reg * W)

    change_b = (learning_rate * np.sum(gradient, axis=0))/m

    #acum acutalizez vitezele folosind inertia trecuta pe coborare pe panta
    V_w = beta*V_w + change_w
    V_b = beta*V_b + change_b


    #aplic viteza pe ponderi

    W = W + V_w
    b = b + V_b
    return (W,b,V_w,V_b)

print(W)
print()
print(b)

Target = get_target(y_train)
(W,b,V_w,V_b) = gradient_descend(Target,Y_prob,0.01,X_train,W,b,V_w,V_b,beta)

print(W)
print()

print(b)


[[-1.56627765e-01  1.67734863e-01  6.00705536e-01 ...  2.35273726e-01
   2.90750967e-01 -1.33548326e-03]
 [ 6.90006613e-03  5.77677182e-03 -6.01388823e-05 ... -7.99094059e-04
  -6.36989498e-04 -4.83220533e-03]
 [-2.17294235e-01 -1.41779483e-01 -4.94383300e-02 ...  9.34518137e-02
   9.43783202e-02  8.45355908e-02]
 ...
 [-1.79166777e-01 -9.63286579e-02  2.61783530e-02 ... -3.24170036e-02
   1.32558524e-01 -4.43419297e-02]
 [-2.42790204e-01 -5.95066759e-02  3.81295319e-02 ...  2.66939733e-02
  -3.04583620e-02  1.94853212e-01]
 [ 7.38381510e-03 -5.52045381e-02  3.73839905e-03 ... -6.99322071e-02
   1.68400324e-01  4.35122284e-02]]

[ 1.3679439  -0.35752447  1.06501987 -0.91762479  0.05235076 -0.25358573
  0.11487083 -2.36016236 -0.53426729  1.82297927]
[[-1.56897238e-01  1.67452741e-01  6.01302177e-01 ...  2.35564078e-01
   2.91450634e-01 -1.63596108e-03]
 [ 6.90064093e-03  5.77651887e-03 -6.03293509e-05 ... -7.99205872e-04
  -6.37102814e-04 -4.83195494e-03]
 [-2.17342640e-01 -1.41835305e

Acum trecem la loop de antrenare

In [20]:
W = np.random.randn(784, 10) * np.sqrt(1.0 / 784) # weight init Xavier/Glorot -> Variatia pondeirlor trb as fie invers prop cu nr de neuroni
b = np.zeros(10)

V_w = np.zeros((784, 10)) #viteza pondere pt leraning rate momentum
V_b = np.zeros(10) # viteza pentru bias
beta = 0.9 # coeficient de frecare

epoches = 120
learning_rate = 0.05
batch_size = 128
Target = get_target(y_train)

print("Starting training...")

for epoch in range(0,epoches+1):

    if epoch == 70:
        learning_rate = 0.001
        
    permutation = np.random.permutation(X_train.shape[0])

    X_shuffled = X_train[permutation]
    Y_shuffled = y_train[permutation]

    for i in range(0,X_train.shape[0],batch_size):
        X_batch = X_shuffled[i:i+batch_size]
        Y_batch = Y_shuffled[i:i+batch_size]

        Target_batch = get_target(Y_batch)

        Y_prob_batch = forward_propagation(X_batch,W,b)
        W, b, V_w, V_b = gradient_descend(Target_batch, Y_prob_batch, learning_rate, X_batch, W, b, V_w, V_b,beta)
    #printuri generale

    if epoch % 10 == 0:
        Y_prob_full = forward_propagation(X_train, W, b)
        loss = cross_entropy(Y_prob_full, y_train)
        Y_hat = predicted_class_matrix(Y_prob_full)
        accuracy = np.mean(Y_hat.flatten() == y_train)
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Accuracy: {accuracy * 100:.2f}%")

print("Finished training")

Starting training...
Epoch 000 | Loss: 0.3672 | Accuracy: 89.77%
Epoch 010 | Loss: 0.2800 | Accuracy: 92.29%
Epoch 020 | Loss: 0.2743 | Accuracy: 92.31%
Epoch 030 | Loss: 0.2676 | Accuracy: 92.55%
Epoch 040 | Loss: 0.2641 | Accuracy: 92.66%
Epoch 050 | Loss: 0.2632 | Accuracy: 92.74%
Epoch 060 | Loss: 0.2628 | Accuracy: 92.72%
Epoch 070 | Loss: 0.2605 | Accuracy: 92.76%
Epoch 080 | Loss: 0.2588 | Accuracy: 92.85%
Epoch 090 | Loss: 0.2587 | Accuracy: 92.83%
Epoch 100 | Loss: 0.2586 | Accuracy: 92.84%
Epoch 110 | Loss: 0.2586 | Accuracy: 92.82%
Epoch 120 | Loss: 0.2586 | Accuracy: 92.86%
Finished training


Testam datele

In [16]:
import pandas as pd

Y_prob_test = forward_propagation(X_test, W, b)

Y_predicted_test = predicted_class_matrix(Y_prob_test)
submission = pd.DataFrame({
    "ID": range(len(Y_predicted_test)),
    "target": Y_predicted_test.flatten()
})

submission.to_csv("submission.csv", index=False)
print("Submission file created")


Submission file created
